# Lesson 3.1: The ReAct loop in 40 lines
*Build an AI Agent from Scratch with LangGraph · Chapter 3*

**Goal:** turn chapter 2's single tool call into a loop that thinks, acts, observes and repeats until the job is done.

**What you will be able to do**
- name the three parts of one step (think, act, observe) and the two ways the loop stops
- write the `react()` loop yourself: about 20 lines, four tools, one Brightcart incident
- read a trace step by step and spot two risky habits (writing before reading, repeated reads)
- avoid the four traps: no step limit, a missing assistant message, crashing on tool errors, writing first

**How this notebook runs.** Every model reply is saved in a small record/replay cache (`cache_l3_01/`).
If a reply is in the cache, it comes back from disk with no API call. Only one cell is marked
**calls the API**; with the saved cache it runs for free, and without a key it still replays.

In [1]:
# Setup: find the course repo's nightwatch/ folder, load .env, check the key.
import os, sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

for cand in ["../../nightwatch", "../../../nightwatch", "nightwatch"]:   # relative to this notebook
    if (Path(cand) / "agent_v0.py").exists():
        NW = Path(cand).resolve()
        break
else:
    raise SystemExit("Can't find nightwatch/. Open this notebook from the course repo (lessons/<lesson>/notebook/).")
sys.path.insert(0, str(NW))

load_dotenv(find_dotenv(usecwd=True))          # the .env at the repo root, if there is one
HAS_KEY = bool(os.environ.get("GROQ_API_KEY"))
print("nightwatch/ found:", (NW / "agent_v0.py").exists(), "| Python", sys.version.split()[0])
if HAS_KEY:
    print("GROQ_API_KEY loaded.")
else:
    print("No GROQ_API_KEY. Get a free key as in lesson 0.2 (console.groq.com -> API Keys) and put")
    print("GROQ_API_KEY=... in the .env file at the repo root.")
print("Cells marked 'calls the API' need the key only for replies that are not in the cache yet.")

nightwatch/ found: True | Python 3.13.16
GROQ_API_KEY loaded.
Cells marked 'calls the API' need the key only for replies that are not in the cache yet.


## 1. Recall: one tool call (chapter 2)

One tool call is a round trip of three moves: the model **asks** for a tool, your code **runs** it, and the result
goes back as a `tool` message with the **same id**. Today we make that trip again and again.

In [2]:
# The three messages of one round trip, as plain dicts (no API call)
ask = {"role": "assistant", "content": "", "tool_calls": [
    {"id": "call_1", "type": "function", "function": {"name": "check_health", "arguments": "{}"}}]}
result = {"role": "tool", "tool_call_id": "call_1", "content": '{"healthy": false}'}
print(ask["tool_calls"][0]["function"]["name"], "->", result["content"],
      "| same id:", ask["tool_calls"][0]["id"] == result["tool_call_id"])

check_health -> {"healthy": false} | same id: True


## 2. Key words

| Word | Meaning |
|---|---|
| agent loop | calls the model again and again, and runs the tools it asks for in between |
| ReAct | reason + act: the model thinks, acts, then thinks again |
| step | one trip around the loop: one model call plus the tools it asked for |
| observation | what a tool returned; the model reads it on the next step |
| stop condition | ends the loop: the model answers, or we hit a limit |
| trace | the printed record of every step |

## 3. The idea, then a hand trace

Each step: **think** (the model reads the whole message list and decides), **act** (your code runs the tools it asked for),
**observe** (each result goes back into the list as a `tool` message). A reply with **no tool call** is the answer, and the
loop stops. In case it never answers, we cap the steps (here at 8). There is **no plan up front**: at step 1 nothing in the
list said "roll back payments"; each clue goes on the page before the next question.

The recorded run from the video (incident `upstream_deploy-0`, pager: *checkout error rate above 5%*):

| step | tool (args) | what came back | what the model knew after |
|---|---|---|---|
| 1 | `check_health({'': ''})` | healthy False, unhealthy: checkout, payments | two services are down |
| 2 | `recent_deploys({'': ''})` | payments v1.3.5 at 04:12, more below | payments changed just before |
| 3 | `rollback_deploy({'service': 'payments'})` | `rollback_deploy(payments) done.` | **a write, before any log read** |
| 4 | `read_logs({'service': 'checkout'})` | 04:13 ERROR checkout upstream payments ... | checkout blames payments |
| 5 | `read_logs({'service': 'payments'})` | INFO lines | nothing new |
| 6 | `read_logs({'service': 'payments'})` | the same INFO lines | **the repeat added nothing, cost a step** |
| 7 | `check_health({'': ''})` | healthy True, unhealthy [] | fixed |
| 8 | no tool call | the answer text | loop stops |

End: healthy True, writes `[('rollback_deploy', 'payments', None)]`: one write in 8 model calls and 7 tool runs.
The odd `{'': ''}` arguments are a model quirk; those two tools ignore their arguments.

## 4. Write it yourself

First the shop and four tools (three reads, one write). `world` is our simulated Brightcart shop with one incident.

In [3]:
import json
from brightcart.incidents import fresh, generate
from brightcart.sim import World

INC = next(x for x in generate() if x.id == "upstream_deploy-0")
world = World(fresh(INC))


def tool(name, desc, **props):
    return {"type": "function", "function": {"name": name, "description": desc, "parameters": {
        "type": "object", "properties": {k: {"type": "string"} for k in props}, "required": list(props)}}}


TOOLS = [tool("read_logs", "Last log lines of a service.", service=1),
         tool("recent_deploys", "Deploys in the last hour."),
         tool("rollback_deploy", "Roll back the latest deploy of a service.", service=1),
         tool("check_health", "Is every service healthy?")]
RUN = {"read_logs": lambda a: world.logs(a["service"], 8), "recent_deploys": lambda a: world.inc.deploys,
       "rollback_deploy": lambda a: world.act("rollback_deploy", a["service"]), "check_health": lambda a: world.health()}
print("alert:", INC.alert, "| tools:", [t["function"]["name"] for t in TOOLS])

alert: checkout error rate above 5% | tools: ['read_logs', 'recent_deploys', 'rollback_deploy', 'check_health']


The client. In the video it is a plain `OpenAI(...)` client pointed at Groq. Here it is wrapped in a tiny
record/replay cache: the exact request is hashed; a hit is read from disk, a miss calls Groq once and saves the reply.
The loop code does not change.

In [4]:
import hashlib
from openai import OpenAI
from openai.types.chat import ChatCompletion

CACHE = Path("cache_l3_01")
CACHE.mkdir(exist_ok=True)


class CachedClient:
    """client.chat.completions.create(**body) with record/replay: same body -> same saved reply."""
    def __init__(self, key):
        self.real = OpenAI(api_key=key, base_url="https://api.groq.com/openai/v1", max_retries=8) if key else None
        self.chat, self.completions, self.api_calls = self, self, 0

    def create(self, **body):
        h = hashlib.sha256(json.dumps(body, sort_keys=True, default=str).encode()).hexdigest()[:24]
        f = CACHE / f"{h}.json"
        if f.exists():
            saved = json.loads(f.read_text(encoding="utf-8"))
            saved.pop("service_tier", None)        # Groq says "on_demand", which the OpenAI types reject
            return ChatCompletion.model_validate(saved)
        if self.real is None:
            raise RuntimeError("reply not in the cache and no GROQ_API_KEY: see the setup cell")
        out = self.real.chat.completions.create(**body)
        self.api_calls += 1
        f.write_text(out.model_dump_json(), encoding="utf-8")
        return out


client = CachedClient(os.environ.get("GROQ_API_KEY"))
print("cached replies on disk:", len(list(CACHE.glob("*.json"))))

cached replies on disk: 6


The loop: about 20 lines. Read it top to bottom: start the list, call the model, stop on a plain answer,
otherwise append the assistant message, run each tool (errors become results), append each result, repeat.

In [5]:
def react(goal, max_steps=8):
    msgs = [{"role": "system", "content": "You are Nightwatch, Brightcart's on-call agent. Fix the problem, "
                                          "check health, then answer."},
            {"role": "user", "content": goal}]
    for step in range(1, max_steps + 1):
        m = client.chat.completions.create(model="openai/gpt-oss-20b", messages=msgs, tools=TOOLS,
                                           temperature=0).choices[0].message
        if not m.tool_calls:                                   # no action asked -> this is the answer
            print(f"step {step}: ANSWER  {m.content}")
            return m.content
        msgs.append({"role": "assistant", "content": m.content or "",
                     "tool_calls": [c.model_dump() for c in m.tool_calls]})
        for c in m.tool_calls:                                 # act, observe, feed the result back
            args = json.loads(c.function.arguments or "{}")
            try:
                out = RUN[c.function.name](args)
            except Exception as e:
                out = f"ERROR: {e}"
            print(f"step {step}: {c.function.name}({args}) -> {str(out)[:90]}")
            msgs.append({"role": "tool", "tool_call_id": c.id, "content": json.dumps(out, default=str)})
    return "stopped: step limit"

In [6]:
# calls the API (only for replies not yet in cache_l3_01/; the saved cache replays this run for free)
world = World(fresh(INC))
answer = react("PAGER: checkout error rate above 5%")
print("\nhealthy at the end:", world.health()["healthy"], "| writes made:", world.actions,
      "| new API calls:", client.api_calls)

step 1: check_health({'': ''}) -> {'healthy': False, 'unhealthy_services': ['checkout', 'payments'], 'queues_over_limit': {}
step 2: recent_deploys({'': ''}) -> [{'t': '04:12', 'service': 'payments', 'version': 'v1.3.5', 'by': 'ci'}, {'t': '04:01', 's
step 3: read_logs({'service': 'checkout'}) -> ['04:10 WARN checkout slow GC pause 120ms', '04:13 ERROR checkout upstream payments return
step 4: rollback_deploy({'service': 'payments'}) -> rollback_deploy(payments) done.
step 5: check_health({'': ''}) -> {'healthy': True, 'unhealthy_services': [], 'queues_over_limit': {}}
step 6: ANSWER  **Action taken**

1. **Identified the issue** – The `checkout` service was experiencing a high error rate (5%+). Logs showed repeated 500 responses from the `payments` service (`POST /payments/v2/charge`).

2. **Checked health** – `checkout` and `payments` were marked unhealthy.

3. **Reviewed recent deployments** – The latest deployment was `payments v1.3.5` at 04:12, which likely introduced the failure.

**This saved run is not the slides' run.** It took 6 steps and read checkout's logs (step 3) *before* the rollback
(step 4), like the live coding take; the slides' run rolled back first and read payments twice. Even at temperature 0
the model does not always pick the same path, which is why later lessons never trust one run. Count its lines honestly:

In [7]:
import inspect
src = inspect.getsource(react).splitlines()
print("react() is", len(src), "lines, prints included")

react() is 21 lines, prints included


## 5. Library version + check

There is no framework yet (LangGraph rebuilds this loop as a graph in chapter 4). The "library" today is the course's
own `nightwatch/agent_v0.py`: the **same loop** with 17 tools and a `finish()` tool. Replay its saved run on the same
incident (no API call) and check it reaches the same outcome as our 20 lines.

In [8]:
import agent_v0
r = agent_v0.run(INC, mode="replay", sample=0)          # replay: every reply from nightwatch/runs/cache
print("agent_v0:", r["stop_reason"], "| steps", r["steps"], "| healthy", r["healthy"], "| writes", r["writes"])
print("same outcome as our loop:", r["healthy"] == world.health()["healthy"]
      and ("rollback_deploy", "payments", None) in r["writes"] and ("rollback_deploy", "payments", None) in world.actions)

agent_v0: finish | steps 7 | healthy True | writes [('rollback_deploy', 'payments', None)]
same outcome as our loop: True


## 6. Where it shows up in agents

Nightwatch v0 (lesson 3.5) is this loop. Its baseline: 180 saved runs (60 incidents × 3 samples). Count them:

In [9]:
rows = [json.loads(line) for line in open(NW / "runs" / "baseline_gpt-oss-20b.jsonl", encoding="utf-8")]
no_finish = [r for r in rows if not r["final"]]
print("runs", len(rows), "| passed", sum(r["pass"] for r in rows), "| never finished", len(no_finish),
      "| steps used by those:", sorted(set(r["steps"] for r in no_finish)))

runs 180 | passed 124 | never finished 46 | steps used by those: [14]


124 of 180 passed, and the most common failure was not a wrong answer: 46 runs never finished, every one at the
14-step limit. Why a loop does not stop, and how to make it stop on purpose, is lesson 3.2.

## 7. Traps

To show the traps without paying for calls, a **scripted client** plays the model: it returns replies we wrote.

In [10]:
def reply(tool_name=None, args="{}", text=None, i=0):
    """A fake ChatCompletion: one tool call, or a plain text answer."""
    msg = {"role": "assistant", "content": text}
    if tool_name:
        msg["tool_calls"] = [{"id": f"call_{i}", "type": "function",
                              "function": {"name": tool_name, "arguments": args}}]
    return ChatCompletion.model_validate({"id": f"r{i}", "object": "chat.completion", "created": 0,
        "model": "scripted", "choices": [{"index": 0, "finish_reason": "stop", "message": msg}]})


class ScriptedClient:
    def __init__(self, make):          # make(i) -> the i-th reply
        self.chat, self.completions, self.make, self.n = self, self, make, 0

    def create(self, **body):
        self.n += 1
        return self.make(self.n)


real_client = client

**Trap 1: no step limit.** A model that never answers keeps the loop running, and every step is a paid call.

In [11]:
client = ScriptedClient(lambda i: reply("check_health", i=i))    # never answers
world = World(fresh(INC))
steps = 0
while True:                       # the trap: no limit (we pull the plug at 50 so the notebook can go on)
    client.chat.completions.create(model="x", messages=[])
    steps += 1
    if steps == 50:
        print("50 model calls and still no answer: every one would be billed")
        break
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):     # hide the 8 trace lines
    out = react("PAGER: checkout error rate above 5%", max_steps=8)
print("with max_steps=8 ->", repr(out), "after", client.n - 50, "calls")
client = real_client

50 model calls and still no answer: every one would be billed
with max_steps=8 -> 'stopped: step limit' after 8 calls


**Trap 2: forgetting to append the assistant message.** Then the `tool` message answers a call id the list has never
seen. Groq (like OpenAI) rejects that request. A small check finds it before you send:

In [12]:
def orphan_tool_ids(msgs):
    asked = set()
    bad = []
    for m in msgs:
        for c in m.get("tool_calls") or []:
            asked.add(c["id"])
        if m["role"] == "tool" and m["tool_call_id"] not in asked:
            bad.append(m["tool_call_id"])
    return bad

good = [{"role": "user", "content": "PAGER"}, ask, result]
broken = [{"role": "user", "content": "PAGER"}, result]          # assistant message forgotten
print("good list, orphan ids:", orphan_tool_ids(good))
print("broken list, orphan ids:", orphan_tool_ids(broken))

good list, orphan ids: []
broken list, orphan ids: ['call_1']


In [13]:
# calls the API (one tiny request; skipped without a key) - what the provider says about the broken list
if HAS_KEY:
    try:
        OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1").chat.completions.create(
            model="openai/gpt-oss-20b", messages=broken, max_completion_tokens=20)
        print("accepted (unexpected)")
    except Exception as e:
        print(type(e).__name__, "->", str(e)[:160])
else:
    print("skipped: no GROQ_API_KEY")

BadRequestError -> Error code: 400 - failed to template request: failed to render tokenized output: failed to render tokens with harmony: HarmonyError: EncodingError: Message=rend


**Trap 3: letting a tool error crash the loop.** The error should go back to the model as the result.

In [14]:
world = World(fresh(INC))
try:
    RUN["read_logs"]({"service": "email_outbox"})          # not a service: the tool raises
except Exception as e:
    print("without try, the loop would die here:", type(e).__name__, e)
try:
    out = RUN["read_logs"]({"service": "email_outbox"})
except Exception as e:
    out = f"ERROR: {e}"
print("inside react, the model just reads:", out)

without try, the loop would die here: KeyError "unknown service 'email_outbox'"
inside react, the model just reads: ERROR: "unknown service 'email_outbox'"


**Trap 4: writing before reading.** In the video's trace the rollback came at step 3, before any log line.

In [15]:
hook_trace = [(1, "check_health"), (2, "recent_deploys"), (3, "rollback_deploy"), (4, "read_logs"),
              (5, "read_logs"), (6, "read_logs"), (7, "check_health")]
first_write = next(s for s, t in hook_trace if t == "rollback_deploy")
first_log = next(s for s, t in hook_trace if t == "read_logs")
print(f"first write at step {first_write}, first log read at step {first_log}: "
      f"{'wrote before reading' if first_write < first_log else 'read first'}")

first write at step 3, first log read at step 4: wrote before reading


It got lucky. A real agent should read first; the Nightwatch system prompt says so from lesson 3.5 on.

## 8. Exercises

Each check cell works without the API (the cache and scripted clients cover it).

In [16]:
def check(n, test):
    """Run the asserts in test(). Print one clear line instead of stopping the notebook."""
    try:
        test()
    except AssertionError as e:
        print(f"✗ Exercise {n} failed: {e}")
    except Exception as e:
        print(f"✗ Exercise {n} failed: {type(e).__name__}: {e}")
    else:
        print(f"✓ Exercise {n} passed")

**Exercise 1 (the video's exercise).** Copy `react` into `react_counting(goal, max_steps=3)`. At the start of every
step, append `len(msgs)` to the global list `counts`. Run it on the same pager alert with a fresh `world`, store the
return value in `ans3`. Predict `counts` before you run it.

In [17]:
counts, ans3 = [], None
# YOUR CODE HERE

In [18]:
def test1():
    assert ans3 == "stopped: step limit", f"react should run out of steps with max_steps=3, got {ans3!r}"
    assert counts == [2, 4, 6], f"expected 2, 4, 6 messages (system + goal, then +2 per one-tool step), got {counts}"
check(1, test1)

✗ Exercise 1 failed: react should run out of steps with max_steps=3, got None


**Exercise 2.** Write `history_ok(msgs)`: True when every `tool` message answers a `tool_call_id` that an **earlier**
assistant message asked for, otherwise False.

In [19]:
def history_ok(msgs):
    # YOUR CODE HERE
    pass

In [20]:
def test2():
    a = {"role": "assistant", "content": "", "tool_calls": [{"id": "c1", "type": "function",
         "function": {"name": "check_health", "arguments": "{}"}}]}
    t = {"role": "tool", "tool_call_id": "c1", "content": "{}"}
    assert history_ok([{"role": "user", "content": "x"}, a, t]) is True, "assistant then tool should be OK"
    assert history_ok([{"role": "user", "content": "x"}, t]) is False, "a tool message with no assistant call is not OK"
    assert history_ok([{"role": "user", "content": "x"}, t, a]) is False, "the call must come BEFORE its result"
    assert history_ok([{"role": "user", "content": "x"}]) is True, "no tool messages is fine"
check(2, test2)

✗ Exercise 2 failed: assistant then tool should be OK


**Exercise 3.** Write `safe_run(name, args)`: run `RUN[name](args)` and return its output; if anything raises
(an unknown service, or a tool name that is not in `RUN`), return the text `"ERROR: ..."` instead.

In [21]:
def safe_run(name, args):
    # YOUR CODE HERE
    pass

In [22]:
def test3():
    global world
    world = World(fresh(INC))
    assert isinstance(safe_run("check_health", {}), dict), "check_health should return its dict"
    assert str(safe_run("read_logs", {"service": "email_outbox"})).startswith("ERROR"), "unknown service -> ERROR text"
    assert str(safe_run("restart_everything", {})).startswith("ERROR"), "unknown tool -> ERROR text, not a crash"
check(3, test3)

✗ Exercise 3 failed: check_health should return its dict


**Exercise 4.** Write `writes_before_first_read(trace)` for a trace of `(step, tool)` pairs: return the list of write
tools (here only `rollback_deploy`) called **before** the first `read_logs`. If there is no `read_logs`, every write counts.

In [23]:
def writes_before_first_read(trace):
    # YOUR CODE HERE
    pass

In [24]:
def test4():
    assert writes_before_first_read(hook_trace) == ["rollback_deploy"], "the hook trace rolled back at step 3"
    live = [(1, "check_health"), (2, "recent_deploys"), (3, "read_logs"), (4, "rollback_deploy"), (5, "check_health")]
    assert writes_before_first_read(live) == [], "the live take read checkout's logs first"
    assert writes_before_first_read([(1, "rollback_deploy"), (2, "rollback_deploy")]) == ["rollback_deploy"] * 2
check(4, test4)

✗ Exercise 4 failed: the hook trace rolled back at step 3


## Solutions (try first!)

In [25]:
def react_counting(goal, max_steps=3):
    msgs = [{"role": "system", "content": "You are Nightwatch, Brightcart's on-call agent. Fix the problem, "
                                          "check health, then answer."},
            {"role": "user", "content": goal}]
    for step in range(1, max_steps + 1):
        counts.append(len(msgs))
        m = client.chat.completions.create(model="openai/gpt-oss-20b", messages=msgs, tools=TOOLS,
                                           temperature=0).choices[0].message
        if not m.tool_calls:
            return m.content
        msgs.append({"role": "assistant", "content": m.content or "",
                     "tool_calls": [c.model_dump() for c in m.tool_calls]})
        for c in m.tool_calls:
            args = json.loads(c.function.arguments or "{}")
            try:
                out = RUN[c.function.name](args)
            except Exception as e:
                out = f"ERROR: {e}"
            msgs.append({"role": "tool", "tool_call_id": c.id, "content": json.dumps(out, default=str)})
    return "stopped: step limit"


counts = []
world = World(fresh(INC))          # fresh shop -> the same requests as the full run -> cache hits
ans3 = react_counting("PAGER: checkout error rate above 5%")
print(ans3, counts)

stopped: step limit [2, 4, 6]


In [26]:
def history_ok(msgs):
    asked = set()
    for m in msgs:
        for c in m.get("tool_calls") or []:
            asked.add(c["id"])
        if m["role"] == "tool" and m["tool_call_id"] not in asked:
            return False
    return True

In [27]:
def safe_run(name, args):
    try:
        return RUN[name](args)
    except Exception as e:
        return f"ERROR: {e}"

In [28]:
def writes_before_first_read(trace):
    out = []
    for step, t in trace:
        if t == "read_logs":
            break
        if t == "rollback_deploy":
            out.append(t)
    return out